<a href="https://colab.research.google.com/github/obimahregina-sudo/ycit-200-bixi-analysis/blob/main/Regina_Obimah_BIXI_Revenue_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating a Synthetic Revenue Model & Exploratory Data Analysis on BIXI Data
**YCIT 200 - AI and Data Science in Practice**  
**Student:** Regina Obimah  
**Track:** Python (pandas / NumPy)

This notebook reconstructs BIXI Montreal's 2019 revenue from behavioural trip logs (no financial ledger exists in the data), audits the current features for predicting trip duration, and computes station distances for Station 4000. All pricing rules come from the assignment's Table 1 (Descriptive Fees). Every inference needed to bridge the 'context gap' in the logs is documented in the Assumptions Log (Section 3).

In [7]:
!unzip -o biximontrealrentals2019.zip

Archive:  biximontrealrentals2019.zip
  inflating: OD_2019-04.csv          
  inflating: OD_2019-05.csv          
  inflating: OD_2019-06.csv          
  inflating: OD_2019-07.csv          
  inflating: OD_2019-08.csv          
  inflating: OD_2019-09.csv          
  inflating: OD_2019-10.csv          
  inflating: Stations_2019.csv       


In [8]:
!unzip -o biximontrealrentals2019.zip

Archive:  biximontrealrentals2019.zip
  inflating: OD_2019-04.csv          
  inflating: OD_2019-05.csv          
  inflating: OD_2019-06.csv          
  inflating: OD_2019-07.csv          
  inflating: OD_2019-08.csv          
  inflating: OD_2019-09.csv          
  inflating: OD_2019-10.csv          
  inflating: Stations_2019.csv       


## 1. Load the 2019 data
Files used (BIXI open data, as provided on MyCourses): `OD_2019-04.csv` ... `OD_2019-10.csv` (one file per month, April-October 2019) and `Stations_2019.csv`. Place the CSVs in the same folder as this notebook (or a `data/` subfolder) before running.

In [9]:
import pandas as pd, numpy as np, glob, math

files = sorted(glob.glob('data/OD_2019-*.csv')) or sorted(glob.glob('OD_2019-*.csv'))
trips = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
stations = pd.read_csv('data/Stations_2019.csv') if glob.glob('data/Stations_2019.csv') else pd.read_csv('Stations_2019.csv')

print('Trips:', len(trips))
print('Stations:', len(stations))
trips.head()

/tmp/ipykernel_1360/3904058260.py:4: DtypeWarning: Columns (1,3) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


Trips: 5597845
Stations: 619


,start_date,start_station_code,end_date,end_station_code,duration_sec,is_member
0,2019-04-14 07:55:22,6001,2019-04-14 08:07:16,6132,713,1
1,2019-04-14 07:59:31,6411,2019-04-14 08:09:18,6411,587,1
2,2019-04-14 07:59:55,6097,2019-04-14 08:12:11,6036,736,1
3,2019-04-14 07:59:57,6310,2019-04-14 08:27:58,6345,1680,1
4,2019-04-14 08:00:37,7029,2019-04-14 08:14:12,6250,814,0


## 2. EDA checks before modelling
The brief warns that outliers (trips of 24 hours or even days) can distort a revenue model, so the duration range is verified first.

In [10]:
trips['duration_min'] = trips['duration_sec'] / 60

print('Duration (sec): min', trips.duration_sec.min(), '| max', trips.duration_sec.max())
print('Trips over 24h:', (trips.duration_sec > 86400).sum())
print('Member trips:', int(trips.is_member.sum()), '| Non-member trips:', int((trips.is_member == 0).sum()))
print('Mean duration - members:', round(trips[trips.is_member==1].duration_min.mean(), 1), 'min')
print('Mean duration - non-members:', round(trips[trips.is_member==0].duration_min.mean(), 1), 'min')

Duration (sec): min 61 | max 7199
Trips over 24h: 0
Member trips: 4592783 | Non-member trips: 1005062
Mean duration - members: 12.2 min
Mean duration - non-members: 19.6 min


**Finding:** the published logs contain no extreme outliers - the shortest trip is 61 seconds and the longest is 7,199 seconds (just under 2 hours). No duration cap is applied. *(This verification is the human-in-the-loop correction documented in the GenAI Audit: a precautionary 24-hour cap was removed once the data showed it was unnecessary.)*

## 3. Inference logic - Assumptions Log
| Category | Logic applied | Justification |
|---|---|---|
| Membership split | Member trips split 85% individual / 10% corporate / 5% multi-user; corporate priced at $82.45 ($97 - 15%). For overage, 95% of member trips use the regular bands, 5% the multi-user bands | The logs only flag is_member; the split reflects an individual-dominated member base |
| Distinct members | Member trips / 100 trips per member per season = 45,928 members | Base revenue needs a member count the logs do not contain; 100 trips/season ~ 2 trips on ~50 riding days |
| Pass attribution | Non-member trips: 70% single trips, 20% 1-day passes, 10% 3-day passes; pass trips converted at 4 trips per 1-day pass and 8 per 3-day pass | Passes are invisible in blind logs, so attribution is proportional; dividing by trips-per-pass avoids overstating pass revenue |
| OPUS discounts | Within single trips: 60% at $2.99, 25% OPUS regular $2.50, 15% OPUS discount $2.00 | OPUS is Montreal's standard transit card; the deeper discount reaches a smaller eligible group |

## 4. Part 1 - Revenue Inference Model
Fee rules from Table 1. Members: $0 up to 45 min; $1.80 for 45-60 min; beyond 60 min, $1.80 + $3 per 15-minute slice (ceiling). Multi-user members: included window is 60 min, then $3 per slice. Non-members: $0 up to 30 min; $1.80 for 30-45 min; beyond 45 min, $1.80 + $3 per slice (ceiling).

In [11]:
def member_fee(d, multi=False):
    if multi:
        return 0.0 if d <= 60 else 3.0 * math.ceil((d - 60) / 15)
    if d <= 45: return 0.0
    if d <= 60: return 1.80
    return 1.80 + 3.0 * math.ceil((d - 60) / 15)

def nonmember_fee(d):
    if d <= 30: return 0.0
    if d <= 45: return 1.80
    return 1.80 + 3.0 * math.ceil((d - 45) / 15)

# Worked checks against Table 1 (a rider 1 minute into a new slice pays the full $3):
assert member_fee(61) == 4.80          # 61-min member trip: $1.80 band + one full slice
assert member_fee(75) == 4.80          # still one slice past 60
assert member_fee(76) == 7.80          # two slices
assert member_fee(61, multi=True) == 3.00
assert nonmember_fee(46) == 4.80
print('Ceiling-logic checks passed.')

Ceiling-logic checks passed.


In [12]:
mem = trips[trips.is_member == 1]
non = trips[trips.is_member == 0]

# Base membership revenue
est_members = len(mem) / 100
base_membership = est_members * (0.85 * 97 + 0.10 * 97 * 0.85 + 0.05 * 255)

# Non-member base revenue (single trips + passes)
NM = len(non)
nonmember_base = NM * 0.70 * (0.60 * 2.99 + 0.25 * 2.50 + 0.15 * 2.00) + (NM * 0.20) / 4 * 5.25 + (NM * 0.10) / 8 * 15.00

# Overage fees
member_overage = 0.95 * mem.duration_min.map(member_fee).sum() + 0.05 * mem.duration_min.map(lambda d: member_fee(d, multi=True)).sum()
nonmember_overage = non.duration_min.map(nonmember_fee).sum()

total = base_membership + nonmember_base + member_overage + nonmember_overage
print(f'Base Membership Revenue:  ${base_membership:,.2f}')
print(f'Non-member trip/pass revenue: ${nonmember_base:,.2f}')
print(f'Usage/Overage Fees: ${member_overage + nonmember_overage:,.2f}  (members ${member_overage:,.2f} + non-members ${nonmember_overage:,.2f})')
print(f'TOTAL RECONSTRUCTED REVENUE: ${total:,.2f}')

Base Membership Revenue:  $4,751,004.37
Non-member trip/pass revenue: $2,365,212.40
Usage/Overage Fees: $564,367.38  (members $75,910.38 + non-members $488,457.00)
TOTAL RECONSTRUCTED REVENUE: $7,680,584.16


## 5. Part 2 - Predictive Data Strategy (evidence)
Top predictors in the current schema: **trip distance** (start-to-end station distance), **is_member**, and **start time** (hour / weekend). The cell below quantifies the first two signals.

In [13]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin(np.radians(lat2 - lat1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

lat = stations.set_index('Code')['latitude']; lon = stations.set_index('Code')['longitude']
trips['trip_km'] = haversine(trips.start_station_code.map(lat), trips.start_station_code.map(lon),
                             trips.end_station_code.map(lat), trips.end_station_code.map(lon))
print('Correlation trip distance vs duration:', round(trips[['trip_km', 'duration_min']].corr().iloc[0, 1], 2))
print('Round trips (same start/end station) mean duration:',
      round(trips[trips.start_station_code == trips.end_station_code].duration_min.mean(), 1), 'min')

Correlation trip distance vs duration: 0.65
Round trips (same start/end station) mean duration: 21.9 min


**Strategic roadmap (new features):** (1) Weather at trip start (Environment Canada hourly join) - rain/cold change trip length and demand, improving both duration forecasts and rebalancing. (2) Elevation delta between start and end stations - Montreal rides are directional; elevation improves duration estimates and predicts station drain/fill. (3) Rider history from BIXI's internal account system (trips in last 30 days, usual stations, tenure, pass type) - a rider's own history is the best predictor of their next trip and supports converting frequent casual riders to members. Privacy note: rider-history features carry re-identification risk and must be aggregated/pseudonymized (Quebec Law 25); weather and elevation describe places, not people.

## 6. Part 3 - Distances from Station 4000
Station 4000 is Jeanne-d'Arc / Ontario. Haversine distances to all other 618 stations, sorted.

In [14]:
s4000 = stations[stations.Code == 4000].iloc[0]
others = stations[stations.Code != 4000].copy()
others['distance_km'] = haversine(s4000.latitude, s4000.longitude, others.latitude, others.longitude)
others = others.sort_values('distance_km')
print('CLOSEST:'); print(others.head(3)[['Code', 'name', 'distance_km']].to_string(index=False))
print('\nFURTHEST:'); print(others.tail(3).iloc[::-1][['Code', 'name', 'distance_km']].to_string(index=False))

CLOSEST:
 Code                 name  distance_km
 6398 Desjardins / Ontario     0.193673
 6394     Valois / Ontario     0.327102
 6397   Marché Maisonneuve     0.434361

FURTHEST:
 Code                           name  distance_km
 7105 St-Joseph / 34e avenue Lachine    17.556546
 7107 19e avenue / St-Joseph Lachine    16.912479
 6714           LaSalle / 90e avenue    16.750434


## 7. GenAI Collaboration Audit
**Systems used:** Muse (Cirina, Muse Spark 1.3) for computation support and drafting; this notebook (pandas/NumPy) as the analysis environment.  
**Human-in-the-loop correction:** the AI's first model included a precautionary cap limiting billable duration to 24 hours, based on the brief's outlier warning. I verified the actual duration range in the logs (61 to 7,199 seconds) and removed the cap - it changed nothing and would have misrepresented the data. The ceiling math was also hand-checked on the 61-minute case ($4.80) before the totals were trusted.  
**Disclosure:** AI assisted with code and drafting; all pricing rules come from Table 1, all results were recomputed from the data in this notebook, and the inference assumptions are documented in Section 3. I am responsible for the final submission.